<h3>A simple, nice nerual network</h3>

In [1]:
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

### Blobs - some blob data

In [2]:
def f(x):
    # y = mx + b
    return 0.3*x + 0.2


class Point:
    def __init__(self, width, height):
        self.x = np.random.uniform(-1, 1, width)
        self.y = np.random.uniform(-1, 1, height)
        self.bias = 1
        self.labels = []
    
    def point(self):
        for x, y in zip(self.x, self.y):
            lineY = f(x)
            if y > lineY:
                self.labels.append(1)
            else:
                self.labels.append(-1)

        return self.labels


points = Point(500, 500)
labels = points.point()
x = points.x
y = points.y
inputs = pd.DataFrame({'x':x, 'y':y})

# NeuralNetwork

In [3]:
class NeuralNetwork:
    def __init__(self, numInput, numHidden, numOutput):
        self.lr = 0.1
        self.input_nodes = numInput
        self.hidden_nodes = numHidden
        self.output_nodes = numOutput

        self.weights_ih = np.random.uniform(-1, 1, (self.hidden_nodes, self.input_nodes))
        self.weights_ho = np.random.uniform(-1, 1, (self.output_nodes, self.hidden_nodes))

        self.bias_h = np.random.uniform(-1, 1, (self.hidden_nodes, 1))
        self.bias_o = np.random.uniform(-1, 1, (self.output_nodes, 1))

    def sigmoid(self, x):
        return 1/(1 + np.exp(-x))

    def dsigmoid(self, x):
        return x * (1 - x)

    
    def feed_forward(self, inputs):
        inputs = np.array(inputs)
        # HIDDEN LAYER
        self.hidden_sum = np.matmul(self.weights_ih, np.transpose(inputs)) + self.bias_h
        self.hidden_out = self.sigmoid(self.hidden_sum)
        # OUTPUT LAYER
        self.output_sum = np.matmul(self.weights_ho, self.hidden_out) + self.bias_o
        self.output_out = self.sigmoid(self.output_sum)

        return np.transpose(self.output_out)

    
    def train(self, inputs, targets):
        targets = np.array(targets).reshape(1, -1)
        inputs = np.array(inputs)

        self.feed_forward(inputs)

        # Calculate the output deltas
        output_deltas = (targets - self.output_out) * self.dsigmoid(self.output_out)

        # Hidden layer errors and deltas
        hidden_errors = np.matmul(self.weights_ho.T, output_deltas)
        hidden_deltas = hidden_errors * self.dsigmoid(self.hidden_out)

        # Update weights
        self.weights_ho += self.lr * np.matmul(output_deltas, self.hidden_out.T)
        self.weights_ih += self.lr * np.matmul(hidden_deltas, inputs)
        # Bias weights
        self.bias_h += self.lr * np.sum(hidden_deltas, axis=1, keepdims=True)
        self.bias_o += self.lr * np.sum(output_deltas, axis=1, keepdims=True)

In [4]:
nn = NeuralNetwork(2, 2, 1)

#### Weights

In [5]:
nn.weights_ih

array([[ 0.93556015, -0.03746735],
       [-0.9733445 , -0.19579139]])

In [6]:
nn.weights_ho

array([[0.97621391, 0.43777413]])

### Feed Forward Check

In [7]:
nn.feed_forward(inputs.iloc[0])

array([[0.78360231],
       [0.84459323]])

## Train 

In [8]:
nn.train(inputs, labels)

Worked!

#### Test

In [9]:
preds = nn.feed_forward(inputs)

In [10]:
             # 2x500                  # 500x2               # 4x4
nn.dsigmoid(nn.hidden_out) @ np.transpose(nn.hidden_out)

array([[ 5.47297176, 12.67687709],
       [ 9.55301296, 36.31060501]])

In [11]:
nn = NeuralNetwork(2, 2, 1)

In [12]:
xor = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [1, 1, 0],
    [0, 0, 0]
])

for i in range(100000):
    np.random.shuffle(xor)
    xor_inputs = np.array(xor[:, :2])
    xor_labels = np.array(xor[:, 2:3]).reshape(4)

    nn.train(xor_inputs, xor_labels)

In [13]:
xor = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [1, 1, 0],
    [0, 0, 0]
])

In [14]:
print(f"Inputs: {xor_inputs}")
print(f"Labels: {xor_labels}")
nn.feed_forward(xor_inputs)

Inputs: [[0 1]
 [1 1]
 [1 0]
 [0 0]]
Labels: [1 0 1 0]


array([[0.98873262],
       [0.01176846],
       [0.98887807],
       [0.01292402]])